In [ ]:
import pandas as pd
import textwrap
import re

In [6]:


def _parse_duration_hours(raw: str) -> float | None:
    """Convert heterogeneous duration strings → approximate hours (float)."""
    if not isinstance(raw, str) or not raw.strip():
        return None
    s = raw.lower().strip()

    # patterns: "16 hours", "5 hours a day for 3 days", "12 weeks", "1 year",
    #           "one semester", "35 videos roughly 50 mins each",
    #           "16 videos roughly 4–17 min each", "5hrs", "5hours", "4 hrs"
    patterns = [
        (r"(\d+(?:\.\d+)?)\s*hours?\s*a\s*day\s*for\s*(\d+)\s*days?", lambda m: float(m.group(1)) * int(m.group(2))),
        (r"(\d+(?:\.\d+)?)\s*hrs?\b", lambda m: float(m.group(1))),
        (r"(\d+(?:\.\d+)?)\s*hours?\b", lambda m: float(m.group(1))),
        (r"(\d+)\s*videos?\s*roughly\s*([\d.]+)\s*[-–]\s*([\d.]+)\s*min",
         lambda m: int(m.group(1)) * (float(m.group(2)) + float(m.group(3))) / 2 / 60),
        (r"(\d+)\s*videos?\s*roughly\s*([\d.]+)\s*min", lambda m: int(m.group(1)) * float(m.group(2)) / 60),
        (r"(\d+(?:\.\d+)?)\s*min(?:utes?)?\b", lambda m: float(m.group(1)) / 60),
        (r"(\d+)\s*weeks?\b", lambda m: float(m.group(1)) * 5),     # ~5 hrs/week light estimate
        (r"one\s+semester|a\s+semester", lambda m: 45.0),
        (r"half\s+a\s+semester", lambda m: 22.5),
        (r"(\d+)\s*months?\b", lambda m: float(m.group(1)) * 10),
        (r"(\d+)\s*years?\b", lambda m: float(m.group(1)) * 120),
    ]
    for pattern, fn in patterns:
        m = re.search(pattern, s)
        if m:
            try:
                return round(fn(m), 1)
            except Exception:
                continue
    return None

RAW_COLS = {
    "Competency domain":         "domain",
    "Focus Areas":               "focus_area",
    "Resource title":            "title",
    "URL":                       "lms_link",
    "Platform / host":           "platform",
    "Resource type":             "resource_type",
    "Stated learning outcomes":  "full_description",
    "Stated prerequisites":      "prerequisites",
    "Length (mins)":             "length_raw",
    "Indicated level":           "level",
    "Intended audience":         "audience",
    "Format type (passive / interactive)": "format",
    "Publication date":          "publication_date",
    "Last updated":              "last_updated",
    "Captions / transcripts":    "captions",
    "Mobile accessible":         "mobile_accessible",
    "Skill area":                "priority_skills",
    "Student journey stage":     "journey_stage",
    "Comments":                  "comments",
}


In [7]:


def load_data(path: str = "Online_curation.csv") -> pd.DataFrame:
    raw = pd.read_csv(path, dtype=str)

    # Forward-fill the hierarchical domain & focus area columns
    raw["Competency domain"] = raw["Competency domain"].replace("", pd.NA).ffill()
    raw["Focus Areas"] = raw["Focus Areas"].replace("", pd.NA).ffill()

    # Rename
    raw = raw.rename(columns=RAW_COLS)

    # Drop rows with no title
    df = raw.dropna(subset=["title"]).copy()
    df = df[df["title"].str.strip() != ""].copy()
    df = df.reset_index(drop=True)
    df["id"] = df.index

    # Clean text columns
    for col in ["domain", "focus_area", "level", "format", "journey_stage",
                 "priority_skills", "platform", "resource_type"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().replace({"nan": "", "N/A": "", "Not Stated": ""})

    # Short description: first 250 chars of learning outcomes
    df["short_description"] = df["full_description"].fillna("").apply(
        lambda s: textwrap.shorten(s.replace("\n", " ").strip(), width=250, placeholder="…")
    )

    # Parse duration
    df["duration_hours"] = df["length_raw"].apply(_parse_duration_hours)

    # Normalize skill tags (may be long sentences — truncate for tags)
    def _skill_tags(s) -> list[str]:
        # Handle non-string types
        if pd.isna(s) or not s:
            return []
        s = str(s)
        if s.strip() in ("", "nan", "None"):
            return []
        # If it looks like a sentence (contains comma-separated brief things), split on comma
        parts = [p.strip() for p in s.split(",") if p.strip()]
        # Truncate very long parts to ~50 chars for display
        return [str(p)[:60] + ("…" if len(str(p)) > 60 else "") for p in parts]

    df["skill_tags"] = df["priority_skills"].fillna("").apply(_skill_tags)

    return df


df = load_data()

NameError: name 're' is not defined